# Avoiding Trajectory Visualization

这个 notebook 只负责可视化 `tools/eval_avoiding_checkpoints.py` 保存出的轨迹数据。

推荐流程：

1. 先用评测脚本生成 JSONL 指标和 `.npz` 轨迹文件。
2. 在这里读取 JSONL，选择一个 `trajectory_path`。
3. 交互式绘制轨迹、查看 mode 分布，并可选保存 PNG。

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle

%matplotlib inline

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RESULT_JSONL = REPO_ROOT / "results" / "avoiding_ddpm_transformer_epoch_curve_best.jsonl"
TRAJECTORY_DIR = REPO_ROOT / "results" / "avoiding_trajectories_npz"

RESULT_JSONL, TRAJECTORY_DIR

## 读取评测结果

`trajectory_path` 来自评测脚本的 JSONL 输出；如果 JSONL 里没有轨迹路径，也可以直接从 `TRAJECTORY_DIR` 扫描 `.npz`。

In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    rows = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


results = read_jsonl(RESULT_JSONL)
if results.empty:
    npz_paths = sorted(TRAJECTORY_DIR.glob("*.npz"))
    results = pd.DataFrame({"trajectory_path": [str(path) for path in npz_paths]})

results

## 选择一个轨迹文件

修改 `row_index` 可以切换不同 seed / epoch 的 rollout。

In [ ]:
row_index = 0

trajectory_path = Path(results.loc[row_index, "trajectory_path"])
trajectory_data = np.load(trajectory_path, allow_pickle=True)

trajectories = trajectory_data["trajectories"]
successes = trajectory_data["successes"].astype(bool)
mode_encoding = trajectory_data["mode_encoding"]
modes = trajectory_data["mode_ids"] if "mode_ids" in trajectory_data else mode_encoding.astype(np.int64).dot(1 << np.arange(mode_encoding.shape[-1]))
checkpoint = str(trajectory_data["checkpoint"])
seed = int(trajectory_data["seed"])
epoch = int(trajectory_data["epoch"])
epoch_label = str(trajectory_data["epoch_label"]) if "epoch_label" in trajectory_data else (epoch if epoch >= 0 else "unknown")

print(trajectory_path)
print("trajectories:", trajectories.shape)
print("success_rate:", float(successes.mean()) if len(successes) else 0.0)
print("checkpoint:", checkpoint)

## 绘图函数

In [ ]:
def trim_trajectory(trajectory):
    valid = np.abs(trajectory).sum(axis=1) > 0
    if not valid.any():
        return trajectory[:0]
    return trajectory[:np.where(valid)[0][-1] + 1]


def summarize_modes(successes, modes):
    if not successes.any():
        return 0.0, []
    unique_modes, counts = np.unique(modes[successes], return_counts=True)
    mode_probs = counts / counts.sum()
    entropy = float(-(mode_probs * (np.log(mode_probs) / np.log(24))).sum())
    mode_summary = sorted(zip(unique_modes, counts), key=lambda item: item[1], reverse=True)
    return entropy, mode_summary


def sample_trajectory_indices(successes, max_trajectories=240, seed=0):
    total = len(successes)
    if max_trajectories <= 0 or total <= max_trajectories:
        return np.arange(total)

    rng = np.random.default_rng(seed)
    success_idx = np.where(successes)[0]
    failure_idx = np.where(~successes)[0]
    n_success = min(len(success_idx), max_trajectories // 2)
    n_failure = max_trajectories - n_success
    chosen_success = rng.choice(success_idx, size=n_success, replace=False) if n_success else np.array([], dtype=int)
    chosen_failure = rng.choice(failure_idx, size=min(len(failure_idx), n_failure), replace=False) if n_failure else np.array([], dtype=int)
    return np.concatenate([chosen_success, chosen_failure])


def plot_avoiding_trajectories(trajectories, successes, modes, seed=None, epoch=None, max_trajectories=240):
    draw_indices = sample_trajectory_indices(successes, max_trajectories=max_trajectories)
    fig, ax = plt.subplots(figsize=(8.5, 8.5), dpi=160)
    ax.set_facecolor("#fbfbf7")

    obstacle_specs = [
        (0.5, -0.1, 0.03, "L1"),
        (0.425, 0.08, 0.025, "L2 top"),
        (0.575, 0.08, 0.025, "L2 bottom"),
        (0.35, 0.26, 0.025, "L3 top"),
        (0.5, 0.26, 0.025, "L3 mid"),
        (0.65, 0.26, 0.025, "L3 bottom"),
    ]
    for x, y, radius, label in obstacle_specs:
        ax.add_patch(Circle((x, y), radius, facecolor="#d73027", edgecolor="#7f0000", alpha=0.75, lw=1.0))
        ax.text(x, y + radius + 0.012, label, ha="center", va="bottom", fontsize=7, color="#7f0000")

    finish_y = -0.1 + 2.5 * 0.18
    ax.add_patch(Rectangle((0.15, finish_y - 0.01), 0.5, 0.02, facecolor="#1a9850", alpha=0.22, edgecolor="#1a9850"))
    ax.axhline(finish_y, color="#1a9850", lw=1.2, ls="--", alpha=0.8)
    ax.text(0.665, finish_y, "finish", va="center", fontsize=8, color="#1a9850")

    for y, label in [(-0.1, "level 1"), (0.08, "level 2"), (0.26, "level 3")]:
        ax.axhline(y, color="#bbbbbb", lw=0.8, ls=":", alpha=0.7)
        ax.text(0.12, y, label, va="center", fontsize=7, color="#777777")

    successful_modes = sorted(set(modes[successes]))
    cmap = plt.get_cmap("tab20", max(1, len(successful_modes)))
    mode_to_color = {mode: cmap(i % cmap.N) for i, mode in enumerate(successful_modes)}

    for idx in draw_indices:
        trajectory = trim_trajectory(trajectories[idx])
        if len(trajectory) < 2:
            continue
        if successes[idx]:
            color = mode_to_color.get(modes[idx], "#377eb8")
            ax.plot(trajectory[:, 0], trajectory[:, 1], color=color, lw=1.15, alpha=0.72, zorder=3)
            ax.scatter(trajectory[-1, 0], trajectory[-1, 1], color=color, s=8, alpha=0.75, zorder=4)
        else:
            ax.plot(trajectory[:, 0], trajectory[:, 1], color="#9e9e9e", lw=0.7, alpha=0.18, zorder=2)

    ax.scatter([0.525], [-0.28], marker="*", s=110, color="#fdae61", edgecolor="#7f3b08", zorder=6)

    success_rate = float(successes.mean()) if len(successes) else 0.0
    entropy, mode_summary = summarize_modes(successes, modes)
    ax.set_title(
        f"Avoiding trajectories | seed={seed}, epoch={epoch}\n"
        f"success={success_rate:.3f}, entropy={entropy:.3f}, n={len(successes)}",
        fontsize=11,
    )
    ax.set_xlabel("x position")
    ax.set_ylabel("y position")
    ax.set_xlim(0.18, 0.72)
    ax.set_ylim(-0.32, 0.42)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#e0e0e0", lw=0.6, alpha=0.65)

    legend_items = [
        Line2D([0], [0], color="#9e9e9e", lw=1.5, alpha=0.45, label="failed rollout"),
        Line2D([0], [0], marker="*", color="w", markerfacecolor="#fdae61", markeredgecolor="#7f3b08", markersize=10, label="start"),
        Line2D([0], [0], color="#1a9850", lw=1.5, ls="--", label="finish line"),
    ]
    for mode, count in mode_summary[:8]:
        legend_items.append(Line2D([0], [0], color=mode_to_color.get(mode, "#377eb8"), lw=2, label=f"mode {mode}: {count}"))
    ax.legend(handles=legend_items, loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, fontsize=8)
    fig.tight_layout()
    return fig, ax


## 绘制当前轨迹文件

In [ ]:
fig, ax = plot_avoiding_trajectories(
    trajectories,
    successes,
    modes,
    seed=seed if seed >= 0 else "unknown",
    epoch=epoch_label,
    max_trajectories=240,
)


## 可选：保存当前图像

In [ ]:
PLOT_DIR = REPO_ROOT / "results" / "avoiding_trajectory_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plot_path = PLOT_DIR / f"{trajectory_path.stem}.png"
fig.savefig(plot_path, bbox_inches="tight")
plot_path